## Information Retrieval 24/25, University of Pisa
### Franco Maria Nardini, Rossano Venturini (francomaria.nardini@isti.cnr.it, rossano.venturini@unipi.it)

----

# Inverted Index

---
![image](imgs/common_crawl.png)

[Common Crawl](https://commoncrawl.org/) maintains a free, open repository of web crawl data that can be used by anyone.

- Over 250 billion pages spanning 17 years
- Free and open corpus since 2007
- A snapshot of 3–5 billion new pages are added each month


---

## C4 Dataset

[C4 (Colossal Clean Crawled Corpus) Dataset](https://huggingface.co/datasets/allenai/c4)

A colossal, cleaned version of Common Crawl's web crawl corpus (from Google). Based on Common Crawl dataset: "https://commoncrawl.org".

We use the processed version of Google's C4 dataset by Allen Institute for AI.
They prepared five variants of the data: `en`, `en.noclean`, `en.noblocklist`, `realnewslike`, and `multilingual (mC4)`.

For reference, these are the sizes of the variants:

- `en`: 305GB
- `en.noclean`: 2.3TB
- `en.noblocklist`: 380GB
- `realnewslike`: 15GB
- `multilingual (mC4)`: 9.7TB (108 subsets, one per language)

The `en.noblocklist` variant is exactly the same as the en variant, except we turned off the so-called "badwords filter", which removes all documents that contain words from the lists at https://github.com/LDNOOBW/List-of-Dirty-Naughty-Obscene-and-Otherwise-Bad-Words.

In [ ]:
!pip install datasets nltk unidecode tqdm

<br><br>

#### We download 4 out of 1024 files of size ~318Mb compressed each

In [1]:
from datasets import load_dataset

c4_subset = load_dataset("allenai/c4", data_files="en/c4-train.0102*-of-01024.json.gz")

<br><br>
#### Get a list of URLs and a list of corresponding documents

In [2]:
urls = [x['url'] for x in c4_subset["train"]]
documents = [x['text'] for doc_id, x in enumerate(c4_subset["train"])]

print(f"Number of documents: {len(urls)}")
print(f"Number of characters: {sum(len(x) for x in documents)} ({sum(len(x) for x in documents)/1024**2})") 

Number of documents: 1425269
Number of characters: 3065881920 (2923.8528442382812)


In [3]:
c4_subset = None

In [4]:
print(urls[0])
print(documents[0][:500], "[...]")

https://americanhealthandbeauty.com/articles/2704/non-surgical-fat-reduction--zerona-vs-zeltiq
Liposuction has remained one of the most popular cosmetic surgeries for years as people turn to their doctors to remove the fat that diet and exercise can't seem to touch. Recently, there has been a trend towards less invasive aesthetic options as lasers and fillers replace facelifts and laser lipo takes center stage with traditional liposuction. There are two devices, both currently undergoing FDA testing, which could replace fat reduction surgery altogether. They are Zerona and Zeltiq, and the [...]


----

## Inverted Index

<br><br>

#### Let's build a simple inverted index



In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import string
from unidecode import unidecode
from tqdm import tqdm
import math

# Download necessary data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

In [ ]:
def get_tokens(doc):   
    # Step 1: Convert document to lowercase
    doc = doc.lower()

    # Step 2: # Step 1: Normalize accents e.g., café vs cafe
    doc = unidecode(doc)
    
    # Step 3: Normalize multiple spaces to a single space
    doc = " ".join(doc.split())
    
    # Step 4: Tokenize the document
    tokens = word_tokenize(doc)
    
    # Step 5: Remove punctuation
    tokens_no_punct = [word for word in tokens if word not in string.punctuation]
    
    # Step 6: Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens_no_stopwords = [word for word in tokens_no_punct if word not in stop_words]
    
    # Step 7: Lemmatization (or stemming)
    #lemmatizer = WordNetLemmatizer()
    #tokens_lemmatized = [lemmatizer.lemmatize(word) for word in tokens_no_stopwords]
    
    # Step 8: Remove numbers
    tokens_no_numbers = [word for word in tokens_no_stopwords if not word.isdigit()]

    return tokens_no_numbers
    

In [ ]:
get_tokens(documents[2])[:10]

<br><br>

<img src="imgs/inverted_indexes.png" alt="alt text" width="700" />

In [ ]:
vocabulary_map = {} # Map from token to its id
df = []             # for ith term, tf[i] stores the number of documents containing this term (red in the image above)
frequencies = []    # for ith term, frequencies[i] stores its frequency in the whole collection (green in the image above)

posting_lists = []  # list of lists. In real life, it is a big list with offsets. (Lists in Python are vectors!)
tf = []             # list of lists. For each document, the frequency of the term in the document

doc_lengths = []    # the length of each document

for doc_id, document in tqdm(enumerate(documents)):
    terms = get_tokens(document)

    doc_lengths.append(len(terms))
    
    for term in terms: 
        if term not in vocabulary_map: # new term
            vocabulary_map[term] = len(vocabulary_map)
            posting_lists.append([])
            tf.append([])
            frequencies.append(0)
            df.append(0)
            
        term_id = vocabulary_map[term]
        frequencies[term_id] += 1
        
        if len(posting_lists[term_id]) == 0 or posting_lists[term_id][-1] != doc_id: # avoid duplicated doc_ids within the same list
            posting_lists[term_id].append( doc_id )
            tf[term_id].append( 0 )
            df[term_id] += 1
            
        tf[term_id][-1] += 1


<br><br>
#### BM25

To create an inverted index with BM25 scores in Python, you can follow these steps:

1. **Understand BM25 Formula**: The BM25 score for a term $ t $ in a document $ d $ is calculated as:

$$
BM25(t, d) = IDF(t) \cdot \frac{f(t, d) \cdot (k_1 + 1)}{f(t, d) + k_1 \cdot (1 - b + b \cdot \frac{|d|}{\text{avgdl}})}
$$

Where:
- $ f(t, d) $ is the frequency of the term $ t $ in the document $ d $.
- $ |d| $ is the length of the document.
- $ \text{avgdl} $ is the average document length in the collection.
- $ k_1 $ and $ b $ are hyperparameters (commonly set as $ k_1 = 1.5 $ and $ b = 0.75 $).
- $ IDF(t) $ is the inverse document frequency of the term $ t $:

$$
IDF(t) = \log\left(\frac{N - df(t) + 0.5}{df(t) + 0.5} + 1\right)
$$

Where $ N $ is the total number of documents and $ df(t) $ is the document frequency of term $ t $ (number of documents containing $ t $).

In [ ]:
# Function to compute IDF for a term
def compute_idf(df_t, N):
    return math.log((N - df_t + 0.5) / (df_t + 0.5) + 1)
    
N = len(df)

# We precompute the idf. Online it could be expensive for the computation of ln 
idf = [ compute_idf(df_t, N) for df_t in df ] 

In [ ]:
# Average document length
avgdl = sum(doc_lengths) / N

# Function to compute BM25 score for a term in a document
# - b is in [0,1]. Larger b favors shorter documents
def compute_bm25(term_id, doc_id, term_freq, idf, avgdl, k1 = 1.5, b = 0.75):
    idf_t = idf[term_id]
    doc_len = doc_lengths[doc_id]
    
    # BM25 formula
    numerator = term_freq * (k1 + 1)
    denominator = term_freq + k1 * (1 - b + b * (doc_len / avgdl))
    
    return idf_t * (numerator/denominator)

<br>

----

#### Save on Disk

In [ ]:
import pickle

In [ ]:
filename = 'inverted_index.pkl' 

In [ ]:
with open(filename, 'wb') as file:
    pickle.dump((N, avgdl, vocabulary_map, posting_lists, df, idf,  frequencies, tf, doc_lengths), file)

#### Read from Disk

In [ ]:
with open(filename, 'rb') as file:
    N, avgdl, vocabulary_map, posting_lists, df, idf, frequencies, tf, doc_lengths = pickle.load(file)

----

#### Checks

In [ ]:
frequencies[:5]

In [ ]:
len(frequencies)

assert len(frequencies) == len(df),  "both must have one element per term"
assert len(frequencies) == len(tf),  "both must have one element per term"

In [ ]:
sorted(vocabulary_map.items(), key = lambda x: x[1])[:5]

In [ ]:
term = "one"

term_id = vocabulary_map[term]

list( zip(posting_lists[term_id], tfs[term_id]) )[:5]

In [ ]:
doc_id = posting_lists[term_id][0]
freq = tfs[term_id][0]

print( freq, documents[doc_id].count(term) )

----

## Query Processing


<br><br>

### Term-at-a-Time (TAAT)


<img src="imgs/taat.png" alt="alt text" width="700" />


In [ ]:
def taat_or(query):
    accumulators = {}
    
    for term in get_tokens(query):
        print(f"Processing term: {term}")
        if term not in vocabulary_map:
            print("Term not present")
            continue
        term_id = vocabulary_map[term]
        
        for doc_id, term_freq in zip(posting_lists[term_id], tf[term_id]):
            score = compute_bm25(term_id, doc_id, term_freq, idf, avgdl)
            if doc_id not in accumulators:
                accumulators[doc_id] = 0.0 
            accumulators[doc_id] += score
            
    return sorted(accumulators.items(), key = lambda x: x[1], reverse = True)


In [ ]:
# Print top documents with their scores
def print_top(top):
    print("\n\n")
    for doc_id, score in top:
        print(f"Score: {score}")
        print(documents[doc_id])
        print("\n\n" + "-"*50 + "\n\n")
        

In [ ]:
top_5 = taat_or("Rust programming language")[:5]

print_top(top_5)

### None of the results is about Rust
Let's implement AND query.

In [ ]:
def taat_and(query):
    accumulators = {}
    for term in get_tokens(query):
        print(f"Processing term: {term}")
        if term not in vocabulary_map:
            print("Term not present")
            continue
        term_id = vocabulary_map[term]
        
        for doc_id, term_freq in zip(posting_lists[term_id], tf[term_id]):
            score = compute_bm25(term_id, doc_id, term_freq, idf, avgdl)
            if doc_id not in accumulators:
                accumulators[doc_id] = (0, 0.0) # number of matched terms, score
            (matches, cur_score) = accumulators[doc_id]
            accumulators[doc_id] = (matches+1, cur_score + score) 
    return sorted(accumulators.items(), key = lambda x: x[1], reverse = True) # sort by number of matches first

In [ ]:
top_5 = taat_and("Rust programming language")[:5]

print_top(top_5)

<br>

### Exercise

Modern search engines allows the use of operator "+" to force the presence of a term.
For example the query "Programming language +Rust" forces the presence of term "Rust". 

Are you able to modify the previous to allows operator "+"?

<br><br>

### Document-at-a-Time (DAAT)

<br><br>

#### Support operations with skipping


<img src="imgs/skipping.png" alt="alt text" width="700" />


In [ ]:
class Skipping:
    def __init__(self, plist, skipping = 100):
        self.skipping = skipping
        self.plist = plist
        self.skips = plist[skipping-1::skipping] # the first element is the skipping-th one

class Operations:
    def __init__(self, skips):
        self.pos = 0
        self.skips = skips
        self.cost = 0  # Number of touched postings

    def get(self):
        return (self.pos, self.docId())

    def next(self):
        self.pos += 1
        return (self.pos, self.docId())

    def nextGEQ(self, target):
        skipping = self.skips.skipping
        
        while self.pos < len(self.skips.plist) and self.skips.plist[self.pos] < target:
            if self.pos % skipping == 0: # I can use skipping
                break
            self.pos += 1
            self.cost += 1 

        pos_skips = self.pos // skipping
        while pos_skips < len(self.skips.skips) and self.skips.skips[pos_skips] < target:
            pos_skips += 1
            self.cost += 1 

        self.pos = pos_skips * skipping
        
        while self.pos < len(self.skips.plist) and self.skips.plist[self.pos] < target:
            self.pos += 1
            self.cost += 1 

        return (self.pos, self.docId())

    def docId(self):
        if self.pos >= len(self.skips.plist):
            return None
        return self.skips.plist[self.pos]

    def get_cost(self):
        return self.cost

##### Example

In [ ]:
l = [0, 10, 11, 20, 21, 22, 23, 28, 30, 35]
#    0   1   2   3   4   5   6   7   8   9

skips = Skipping(l, 5)

ops = Operations(skips)

print(skips.skips)

In [ ]:
ops.nextGEQ(23)

In [ ]:
ops.nextGEQ(30)

In [ ]:
ops.nextGEQ(31)

In [ ]:
ops.nextGEQ(37)

<br><br>

#### AND query with skipping

In [ ]:
def get_terms_posting_list(query):
    term_lists = []     # posting lists of the query terms
    curr_doc_ids = []   # the doc_ids currently pointed on the posting lists
    
    for term in get_tokens(query):
        if term not in vocabulary_map: 
            break
        term_id = vocabulary_map[term]

        curr_list = Operations( Skipping( posting_lists[term_id], skipping) ) #### Here Skipping is built ONLINE. In real life, this is done OFFLINE!
        term_lists.append( curr_list )
        _, doc_id = curr_list.get()
        curr_doc_ids.append( doc_id )  # doc_id at position 0
        
    return term_list, curr_doc_ids

<img src="imgs/daat_and.png" alt="alt text" width="700" />


In [ ]:
def AND_skipping(query, skipping = 100):
    
    term_list, curr_doc_ids = get_terms_posting_list(query)
    
    r = []
    while True:
        max_doc_id = max(curr_doc_ids)
        arg_min, min_doc_id = min( enumerate(curr_doc_ids), key = lambda x: x[1]); # compare by doc_ids 

        doc_id = 0
        
        if max_doc_id == min_doc_id: # we have a result
            
            r.append( max_doc_id )
            _, doc_id = term_lists[0].next()
            curr_doc_ids[0] = doc_id
        else:
            _, doc_id = term_lists[arg_min].nextGEQ( max_doc_id ) 
            curr_doc_ids[arg_min] = doc_id

        if doc_id == None:
            break
            
    cost = sum(term_list.get_cost() for term_list in term_lists)
    return r, cost

In [ ]:
query = "information retrieval python"

for skipping in [1, 5, 10, 15, 20, 50, 75, 100, 150]:
    r, cost = AND_skipping(query, skipping)
    print("skipping: {0:>3}\tcost: {1:>10}\tnumber of results: {2:>5}".format(skipping,cost, len(r)) )
print()

cost = 0
for term in get_tokens(query):
    if term in vocabulary_map: 
        term_id = vocabulary_map[term]
        cost += len(posting_lists[term_id])        
        print("term: {0:13}  n_postings: {1:>10}".format(term, len(posting_lists[term_id])))

print(f"\nWithout skipping the cost would be: {cost}")

<br><br>

#### AND query with shortest lists first

<img src="imgs/taat_and.png" alt="alt text" width="700" />

In [ ]:
def AND_shortest_first(query, skipping = 100):
    
    term_list, curr_doc_ids = get_terms_posting_list(query)

    term_lists.sort(key = lambda x: len(x)) # sort by length

    cost = 0
    r = term_lists[0]
    for cur_list in term_lists[1:]:
        cur_list = Operations( Skipping( cur_list, skipping) ) #### Here Skipping is built ONLINE. In real life, this is done OFFLINE!
        cur_r = []
        for e in r:
            _, doc_id = cur_list.nextGEQ(e)
            if doc_id == None:
                break
            if e == doc_id:
                cur_r.append(e)
        cost += cur_list.get_cost()
        r = cur_r
    return r, cost

In [ ]:
query = "information retrieval python"

for skipping in [1, 5, 10, 15, 20, 50, 75, 100, 150]:
    r, cost = AND_shortest_first(query, skipping)
    print("skipping: {0:>3}\tcost: {1:>10}\tnumber of results: {2:>5}".format(skipping,cost, len(r)) )
print()

cost = 0
for term in get_tokens(query):
    if term in vocabulary_map: 
        term_id = vocabulary_map[term]
        cost += len(posting_lists[term_id])
        print("term: {0:13}  n_postings: {1:>10}".format(term, len(posting_lists[term_id])))

print(f"\nWithout skipping the cost would be: {cost}")

<br><br>

Lower cost! But more esperiments are needed in order to evaluate the fastest in practice!